# 01 · What is a graph?

**Goal:** by the end you can name the three parts of every LangGraph program (**nodes**, **edges** and **state**)
and point to each one in a small calculator written in plain Python.

## The idea in plain English

A program that does a job in steps can be drawn as boxes joined by arrows. Each box is a **node**: one step of work.
Each arrow is an **edge**: it says which step runs next. The steps share a notepad called the **state**. Every step
reads what it needs from the notepad and writes down what it found. The whole drawing (boxes, arrows and notepad) is
a **graph**.

This lesson uses no LangGraph at all. We build a tiny calculator from three ordinary Python functions, run them in
order, and then look at what plain Python makes awkward. Those awkward parts are exactly what LangGraph is for.

**Analogy:** an office where a form is passed from desk to desk. Each clerk (node) fills in a few boxes on the form
(state), then hands it to the next desk the arrows (edges) point to.

## Picture

This is the calculator we are about to write. `START` and `END` are just "the form comes in" and "the form goes out".

```mermaid
flowchart LR
    START([START]) --> parse
    parse --> calculate
    calculate --> format_answer
    format_answer --> END([END])
```

| Step (node) | Reads from the state | Writes to the state |
|---|---|---|
| `parse` | `expression` (`"3 + 4"`) | `a`, `op`, `b` (`3.0`, `"+"`, `4.0`) |
| `calculate` | `a`, `op`, `b` | `result` (`7.0`) |
| `format_answer` | `expression`, `result` | `answer` (`"3 + 4 = 7"`) |

## Step 1: The state is a dict

The state is a plain Python dict. At the start it holds only what the user typed. Each step will add more keys.

In [1]:
state = {"expression": "3 + 4"}
print(state)

{'expression': '3 + 4'}


## Step 2: The first node, `parse`

A node is a function that takes the state and returns **only the new information** it found. It doesn't touch the
rest of the notepad. `parse` splits `"3 + 4"` on spaces into three pieces and turns the numbers into `float`s.

We call it directly to see what it returns. It's just a Python function.

In [2]:
def parse(state):
    """Turn "3 + 4" into a=3.0, op="+", b=4.0."""
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


print(parse(state))

{'a': 3.0, 'op': '+', 'b': 4.0}


## Step 3: The other two nodes

`calculate` reads `a`, `op` and `b` and returns the `result`. `format_answer` builds a sentence for the user.
The `:g` inside the f-string prints `7.0` as `7` and `2.5` as `2.5`.

In [3]:
def calculate(state):
    """Apply the operator to a and b."""
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        result = a + b
    elif op == "-":
        result = a - b
    elif op == "*":
        result = a * b
    elif op == "/":
        result = a / b
    else:
        raise ValueError(f"Unknown operator: {op}")
    return {"result": result}

`format_answer` reads the original text and the result, and builds the sentence the user will see.

In [4]:
def format_answer(state):
    """Build the sentence the user sees."""
    return {"answer": f"{state['expression']} = {state['result']:g}"}

## Step 4: Run the steps by hand and watch the notepad grow

`dict.update` copies the keys a node returned into the notepad. Old keys stay; new keys are added. We print the
state after every step so you can see each node's contribution.

In [5]:
state = {"expression": "3 + 4"}
print("start:              ", state)

state.update(parse(state))
print("after parse:        ", state)

state.update(calculate(state))
print("after calculate:    ", state)

state.update(format_answer(state))
print("after format_answer:", state)

start:               {'expression': '3 + 4'}
after parse:         {'expression': '3 + 4', 'a': 3.0, 'op': '+', 'b': 4.0}
after calculate:     {'expression': '3 + 4', 'a': 3.0, 'op': '+', 'b': 4.0, 'result': 7.0}
after format_answer: {'expression': '3 + 4', 'a': 3.0, 'op': '+', 'b': 4.0, 'result': 7.0, 'answer': '3 + 4 = 7'}


## Step 5: A tiny "graph runner"

Writing `state.update(...)` three times gets old. Below, the list `steps` plays the role of the **edges**: it says
"run `parse`, then `calculate`, then `format_answer`". The function `run_calculator` is a very small engine. It
creates a fresh state, runs each node in order, and merges each update into the state.

This is the core of what LangGraph does for you, minus everything clever.

In [6]:
steps = [parse, calculate, format_answer]  # the "edges": one fixed order


def run_calculator(expression):
    state = {"expression": expression}  # a fresh notepad for every run
    for step in steps:
        state.update(step(state))
    return state


for expression in ["3 + 4", "10 - 2.5", "6 * 7", "10 / 4"]:
    print(run_calculator(expression)["answer"])

3 + 4 = 7
10 - 2.5 = 7.5
6 * 7 = 42
10 / 4 = 2.5


## What plain Python makes awkward

Our runner works for the simple case. Real programs need four more things. Let's try each one and see where the
plain-Python version gets stuck.

### Missing piece 1: branching

Dividing by zero should go to an "error" step and print a friendly message. Our runner only knows "run the next
step in the list", so it simply crashes. The `try/except` below catches the crash so we can print it.

There *is* branching in our code: the `if/elif` inside `calculate`. But it's hidden inside one function. You can't
see it in the list of steps, and you can't send the state to a different step from there.

In [7]:
try:
    run_calculator("10 / 0")
except ZeroDivisionError as err:
    print("Crashed with", type(err).__name__ + ":", err)

Crashed with ZeroDivisionError: float division by zero


### Missing piece 2: looping

"Start at 3 and keep doubling until the result is over 100." The list of steps runs each step once. To repeat a
step we have to write our own `while` loop *around* it, with the exit test and the bookkeeping mixed in. If the exit
test were wrong, this cell would run forever and nothing would stop it.

In [8]:
state = {"a": 3.0, "op": "*", "b": 2.0}
rounds = 0
while True:
    state.update(calculate(state))
    rounds += 1
    if state["result"] > 100:  # the exit test
        break
    state["a"] = state["result"]  # feed the answer back in

print(f"After {rounds} rounds:", state)

After 6 rounds: {'a': 96.0, 'op': '*', 'b': 2.0, 'result': 192.0}


### Missing piece 3: memory

We'd like to type `"3 + 4"` and then `"+ 10"` to mean "add 10 to my last answer". But `run_calculator` starts a
fresh notepad every time, so the second run has never heard of the first. `parse` gets only two pieces and fails.

In [9]:
first = run_calculator("3 + 4")
print("First run:", first["answer"])

try:
    run_calculator("+ 10")
except ValueError as err:
    print("Second run crashed with ValueError:", err)

First run: 3 + 4 = 7
Second run crashed with ValueError: not enough values to unpack (expected 3, got 2)


### Missing piece 4: pausing for a human

Before dividing by zero, we might want to stop and ask a person "what divisor did you mean?", then carry on, maybe
hours later. Python's `input()` can ask a question, but the program then sits blocked, waiting. Nothing saves the
notepad so the run can be resumed tomorrow or on another machine. There's no short code cell for this one, which is
the point: plain Python has no built-in way to do it.

### What LangGraph adds

You still write nodes as plain functions and describe the state. LangGraph runs them, and adds the missing pieces:

| Missing piece | What we'd hand-write | LangGraph feature | Lesson |
|---|---|---|---|
| Branching | `if` statements inside the runner | Conditional edges | 05 |
| Looping | `while` loops plus our own safety stop | Loops and the recursion limit | 07 |
| Memory | Save and reload the notepad ourselves | Checkpointers and threads, the store | 13, 17 |
| Pausing | Save everything, exit, reload, resume | `interrupt` | 14 |

It also draws the graph for you, so the branches and loops are visible instead of buried in `if` and `while`.

## What just happened

- **Step 1** created the state, a dict holding only `{'expression': '3 + 4'}`.
- **Steps 2–3** wrote three nodes. Each takes the state and returns a *small* dict of new keys. `parse` returned
  `{'a': 3.0, 'op': '+', 'b': 4.0}` and nothing else.
- **Step 4** ran the nodes by hand. The printed state grew one step at a time until it held `answer: '3 + 4 = 7'`.
- **Step 5** put the order of the steps (the edges) in a list and wrote a tiny runner that merges each update into
  the state. It printed four answers, including `10 / 4 = 2.5`.
- **The missing pieces** showed where plain Python gets stuck: `10 / 0` crashed instead of branching, looping needed
  a hand-written `while`, `"+ 10"` failed because there's no memory, and pausing for a human has no simple solution.

## Common mistakes

1. **A node forgets to `return`.** A Python function with no `return` gives back `None`, and merging `None` into the
   state fails:
   `TypeError: 'NoneType' object is not iterable`.
   Fix: make every node end with `return {...}`.
2. **Steps in the wrong order.** If `calculate` runs before `parse`, the keys it reads don't exist yet:
   `KeyError: 'a'`.
   Fix: check the order of the edges. Each node can only read what an earlier node wrote.
3. **No spaces in the expression.** `"3+4".split()` gives one piece instead of three, so `parse` fails:
   `ValueError: not enough values to unpack (expected 3, got 1)`.
   Our simple parser needs spaces around the operator: `"3 + 4"`.

## Try it

1. Add a power operator, `**`, to `calculate`, then run `run_calculator("2 ** 10")`. You should get `2 ** 10 = 1024`.
2. Write a fourth node, `round_result`, that rounds `result` to 2 decimal places. Put it in `steps` between
   `calculate` and `format_answer`, then run `run_calculator("10 / 3")`. You should get `10 / 3 = 3.33`.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Graph**: a program drawn as nodes joined by edges, all sharing one state.
- **Node**: one step of work. A function that reads the state and returns an update.
- **Edge**: an arrow saying which node runs next.
- **State**: the shared notepad (a dict) that every node can read and add to.
- **START / END**: the entry and exit points of a graph. In LangGraph they're real names you import (lesson 04).